# This notebook serves as a template for cleaning the original text data and saving the train/test splits to Parquet files rather than csv

In [1]:
# !pip install nltk

# !pip install pyspellchecker

# !pip install textblob

In [4]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [5]:
train.head()

,essay_id,full_text,score
0,000d118,Many people have car where they live. The thin...,3
1,000fe60,I am a scientist at NASA that is discussing th...,3
2,001ab80,People always wish they had the same technolog...,4
3,001bdc0,"We all heard about Venus, the planet without a...",4
4,002ba53,"Dear, State Senator\n\nThis is a letter to arg...",3


In [6]:
test.head()

,essay_id,full_text
0,000d118,Many people have car where they live. The thin...
1,000fe60,I am a scientist at NASA that is discussing th...
2,001ab80,People always wish they had the same technolog...


In [2]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string
from spellchecker import SpellChecker
from textblob import TextBlob

# # Set the download location to Kaggle's working directory
# download_dir = '/kaggle/working/nltk_data'

# # Add this download directory to nltk's data path
# if download_dir not in nltk.data.path:
#     nltk.data.path.append(download_dir)

# Download necessary datasets from NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('omw-1.4')
nltk.download('universal_tagset')
nltk.download('maxent_ne_chunker')
nltk.download('words')

# Now check if the directory is correctly set and files are present
# import os
# print(os.listdir(download_dir))  # This should show the downloaded files


[nltk_data] Downloading package punkt to /home/jack/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/jack/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jack/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/jack/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to /home/jack/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /home/jack/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /home/jack/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker.zip.
[nltk_data] Downloading package 

True

In [8]:
# import zipfile
# import os

# # Path to the zip file
# zip_path = '/kaggle/working/nltk_data/corpora/wordnet.zip'

# # Target extraction directory
# extract_dir = '/kaggle/working/nltk_data/corpora/wordnet'

# # Create the target directory if it doesn't already exist
# if not os.path.exists(extract_dir):
#     os.makedirs(extract_dir)

# # Extract the zip file
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_dir)

# # Verify the files have been extracted
# print(os.listdir(extract_dir))  # Should show the contents of the wordnet corpus


In [9]:
# import os
# import shutil

# # Set the source and target directories
# source_dir = '/kaggle/working/nltk_data/corpora/wordnet/wordnet'
# target_dir = '/kaggle/working/nltk_data/corpora/wordnet'

# # Move each file and subdirectory from the source to the target directory
# for filename in os.listdir(source_dir):
#     source_file = os.path.join(source_dir, filename)
#     target_file = os.path.join(target_dir, filename)
#     if os.path.isdir(source_file):
#         if os.path.exists(target_file):
#             shutil.rmtree(target_file)  # Remove if target directory already exists
#         shutil.move(source_file, target_dir)
#     else:
#         if os.path.exists(target_file):
#             os.remove(target_file)  # Remove if target file already exists
#         shutil.move(source_file, target_dir)

# # Clean up the now empty source directory
# os.rmdir(source_dir)

# # Verify the structure
# print(os.listdir(target_dir))  # Should list 'lexnames' among other files


In [2]:
import os
from tqdm import tqdm

# List of file paths to unzip
embeddings = ['data/archive.zip']

def unzip_embeddings(file_paths):
    """
    Unzip a list of files.

    Args:
        file_paths (list): List of file paths to unzip.

    Returns:
        None
    """
    # Initialize tqdm with the total number of files to unzip
    with tqdm(total=len(file_paths)) as pbar:
        for emb in file_paths:
            # Use the -o flag to automatically replace files
            if os.system(f'unzip -o {emb} -d data/') == 0:
                print(f"Inflating {emb} successful.")
            else:
                print(f"Inflating {emb} failed.")
            pbar.update(1)  # Update the progress bar

# Call the function to unzip files
unzip_embeddings(embeddings)

  0%|          | 0/1 [00:00<?, ?it/s]

Archive:  data/archive.zip
  inflating: data/GoogleNews-vectors-negative-300d.bin  
  inflating: data/glove-840B-300d.txt  
  inflating: data/paragram-300-sl999.txt  
  inflating: data/wiki-news-1M-300d.vec  

100%|██████████| 1/1 [01:40<00:00, 100.67s/it]


Inflating data/archive.zip successful.


In [10]:
# Preprocessing
import numpy as np
import pandas as pd
import operator
from spellchecker import SpellChecker
from tqdm import tqdm  # Import tqdm
import re

def clean_text(df, glove_path, paragram_path, wiki_news_path, col_name = 'full_text'):
    """
    Preprocesses text for both training and testing datasets. 
    Includes loading embeddings, building vocabularies, cleaning text among other things.
    
    :param summaries_train: DataFrame with the training data
    :param summaries_test: DataFrame with the testing data
    :param glove_path: path to the GloVe embedding
    :param paragram_path: path to the Paragram embedding
    :param wiki_news_path: path to the Wiki News embedding
    
    :return: Preprocessed DataFrame and list of out-of-vocab words
    """
    print("Starting text cleaning process.")
    
    def load_embed(file):
        """
        Load the embeddings from a file.
        """
        print(f"Loading embeddings from {file}")
        
        def get_coefs(word, *arr): 
            return word, np.asarray(arr, dtype='float32')
        
        if file == wiki_news_path:
            embeddings_index = dict(get_coefs(*o.split(" ")) for o in tqdm(open(file), "Reading Embedding File") if len(o)>100)
        else:
            embeddings_index = dict(get_coefs(*o.split(" ")) for o in tqdm(open(file, encoding='latin'), "Reading Embedding File"))
        
        print(f"Loaded embeddings from {file}")
        return embeddings_index

    # Load embeddings
    print("Loading all embeddings.")
    embed_glove = load_embed(glove_path)
    embed_paragram = load_embed(paragram_path)
    embed_fasttext = load_embed(wiki_news_path)
    print("All embeddings loaded.")
    
    def build_vocab(texts):
        """
        Build a vocabulary from a given list of texts.
        """
        print("Building vocabulary.")
        sentences = texts.apply(lambda x: x.split()).values
        vocab = {}
        for sentence in tqdm(sentences, "Populating Vocabulary"):
            for word in sentence:
                try:
                    vocab[word] += 1
                except KeyError:
                    vocab[word] = 1
        print("Vocabulary built.")
        return vocab

    def check_coverage(vocab, embeddings_index):
        """
        Check which words in the vocabulary are covered by the embeddings.
        """
        print("Checking coverage.")
        known_words = {}
        unknown_words = {}
        nb_known_words = 0
        nb_unknown_words = 0
        for word in tqdm(vocab.keys(), "Checking Words"):
            try:
                known_words[word] = embeddings_index[word]
                nb_known_words += vocab[word]
            except:
                unknown_words[word] = vocab[word]
                nb_unknown_words += vocab[word]
                pass
        unknown_words = sorted(unknown_words.items(), key=operator.itemgetter(1))[::-1]
        print("Coverage checked.")
        return unknown_words

    # Build and check vocab for train and test datasets
    print("Processing train and test datasets.")
    
    vocab_train = build_vocab(df[col_name])
    
    oov_glove_train = check_coverage(vocab_train, embed_glove)
    oov_paragram_train = check_coverage(vocab_train, embed_paragram)
    oov_fasttext_train = check_coverage(vocab_train, embed_fasttext)
  
    print("Processed train and test datasets.")
    
    # Lowercase all texts
    df['lowered_question'] = df[col_name].apply(lambda x: x.lower())
    
    train_vocab_low = build_vocab(df['lowered_question'])
    
    oov_glove_train = check_coverage(train_vocab_low, embed_glove)
    oov_paragram_train = check_coverage(train_vocab_low, embed_paragram)
    oov_fasttext_train = check_coverage(train_vocab_low, embed_fasttext)
 
    
    def add_lower(embedding, vocab):
        count = 0
        for word in vocab:
            if word in embedding and word.lower() not in embedding:  
                embedding[word.lower()] = embedding[word]
                count += 1
    
    add_lower(embed_glove, train_vocab_low)
    add_lower(embed_paragram, train_vocab_low)
    add_lower(embed_fasttext, train_vocab_low)
   
    
    # Handle contractions
    contraction_mapping = {"ain't": "is not", "aren't": "are not","can't": "cannot", "'cause": "because", "could've": "could have", "couldn't": "could not", 
                           "didn't": "did not",  "doesn't": "does not", "don't": "do not", "hadn't": "had not", "hasn't": "has not", "haven't": "have not", 
                           "he'd": "he would","he'll": "he will", "he's": "he is", "how'd": "how did", "how'd'y": "how do you", "how'll": "how will", "how's": "how is",  
                           "I'd": "I would", "I'd've": "I would have", "I'll": "I will", "I'll've": "I will have","I'm": "I am", "I've": "I have", "i'd": "i would", 
                           "i'd've": "i would have", "i'll": "i will",  "i'll've": "i will have","i'm": "i am", "i've": "i have", "isn't": "is not", "it'd": "it would", 
                           "it'd've": "it would have", "it'll": "it will", "it'll've": "it will have","it's": "it is", "let's": "let us", "ma'am": "madam", 
                           "mayn't": "may not", "might've": "might have","mightn't": "might not","mightn't've": "might not have", "must've": "must have", 
                           "mustn't": "must not", "mustn't've": "must not have", "needn't": "need not", "needn't've": "need not have","o'clock": "of the clock", 
                           "oughtn't": "ought not", "oughtn't've": "ought not have", "shan't": "shall not", "sha'n't": "shall not", "shan't've": "shall not have", 
                           "she'd": "she would", "she'd've": "she would have", "she'll": "she will", "she'll've": "she will have", "she's": "she is", 
                           "should've": "should have", "shouldn't": "should not", "shouldn't've": "should not have", "so've": "so have","so's": "so as", 
                           "this's": "this is","that'd": "that would", "that'd've": "that would have", "that's": "that is", "there'd": "there would", 
                           "there'd've": "there would have", "there's": "there is", "here's": "here is","they'd": "they would", "they'd've": "they would have", 
                           "they'll": "they will", "they'll've": "they will have", "they're": "they are", "they've": "they have", "to've": "to have", 
                           "wasn't": "was not", "we'd": "we would", "we'd've": "we would have", "we'll": "we will", "we'll've": "we will have", "we're": "we are", 
                           "we've": "we have", "weren't": "were not", "what'll": "what will", "what'll've": "what will have", "what're": "what are",  "what's": "what is",
                           "what've": "what have", "when's": "when is", "when've": "when have", "where'd": "where did", "where's": "where is", "where've": "where have",
                           "who'll": "who will", "who'll've": "who will have", "who's": "who is", "who've": "who have", "why's": "why is", "why've": "why have", 
                           "will've": "will have", "won't": "will not", "won't've": "will not have", "would've": "would have", "wouldn't": "would not", 
                           "wouldn't've": "would not have", "y'all": "you all", "y'all'd": "you all would","y'all'd've": "you all would have","y'all're": "you all are",
                           "y'all've": "you all have","you'd": "you would", "you'd've": "you would have", "you'll": "you will", "you'll've": "you will have", 
                           "you're": "you are", "you've": "you have" }
    
    
    def clean_contractions(text, mapping):
        """
        Replace contractions in the text based on a given mapping.
        
        :param text: The original text
        :param mapping: Dictionary containing contractions mapping
        
        :return: Text with contractions replaced
        """
        
        specials = ["’", "‘", "´", "`"]
        
        for s in specials:
            text = text.replace(s, "'")
        text = ' '.join([mapping[t] if t in mapping else t for t in text.split(" ")])
        return text

    # Apply contraction cleaning to train and test datasets
    df['cleaned_text'] = df['lowered_question'].apply(lambda x: clean_contractions(x, contraction_mapping))
    
    # Rebuild and check vocab after cleaning contractions
    vocab_train_clean = build_vocab(df['cleaned_text'])
    
    oov_glove_train = check_coverage(vocab_train_clean, embed_glove)
    oov_paragram_train = check_coverage(vocab_train_clean, embed_paragram)
    oov_fasttext_train = check_coverage(vocab_train_clean, embed_fasttext)
    
     # Add your additional code for punctuations, special characters and spelling correction here
        
    punct = "/-'?!.,#$%\'()*+-/:;<=>@[\\]^_`{|}~" + '""“”’' + '∞θ÷α•à−β∅³π‘₹´°£€\×™√²—–&'
    
    punct_mapping = {"‘": "'", "₹": "e", "´": "'", "°": "", "€": "e", "™": "tm", "√": " sqrt ", "×": "x", "²": "2", "—": "-", "–": "-", "’": "'", "_": "-",
                     "`": "'", '“': '"', '”': '"', '“': '"', "£": "e", '∞': 'infinity', 'θ': 'theta', '÷': '/', 'α': 'alpha', '•': '.', 'à': 'a', '−': '-', 
                     'β': 'beta', '∅': '', '³': '3', 'π': 'pi', }

    def clean_special_chars(text, punct, mapping):
        for p in mapping:
            text = text.replace(p, mapping[p])
        for p in punct:
            text = text.replace(p, f' {p} ')
        specials = {'\u200b': ' ', '…': ' ... ', '\ufeff': '', 'करना': '', 'है': ''}  
        for s in specials:
            text = text.replace(s, specials[s])
        return text

    df['clean_text'] = df['cleaned_text'].apply(lambda x: clean_special_chars(x, punct, punct_mapping))

    # Spell correction
    misspelled_words = [word for word, count in oov_fasttext_train]

    def spell_check_list_of_words(word_list):
        spell = SpellChecker()
        corrected_dict = {}
        for word in word_list:
            corrected_word = spell.correction(word)
            corrected_dict[word] = corrected_word
        return corrected_dict

    mispell_dict_train = spell_check_list_of_words(misspelled_words)

    def correct_spelling(x, dic):
        if not dic:
            return x
        pattern = r'\b(' + '|'.join(re.escape(key) for key in dic.keys()) + r')\b'
        return re.sub(pattern, lambda m: dic[m.group(0)], x, flags=re.IGNORECASE)

    df['clean_text'] = df['clean_text'].apply(lambda x: correct_spelling(x, mispell_dict_train))
    
    # Rebuild and check vocab after cleaning contractions
    vocab_train_clean = build_vocab(df['clean_text'])
    
    oov_glove_train = check_coverage(vocab_train_clean, embed_glove)
    oov_paragram_train = check_coverage(vocab_train_clean, embed_paragram)
    oov_fasttext_train = check_coverage(vocab_train_clean, embed_fasttext)


    return df, oov_glove_train, oov_paragram_train, oov_fasttext_train

In [16]:
# time the run time

import time

start = time.time()

# Clean the train and test text data

train,_,_,_ = clean_text(train, '/home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt', 
           '/home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt', 
           '/home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec', col_name = 'full_text')

# test,_,_,_ = clean_text(test, '/home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt', 
#            '/home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt', 
#            '/home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec', col_name = 'full_text')

end = time.time()

print(f"Time taken to clean text: {end - start} seconds")

Starting text cleaning process.
Loading all embeddings.
Loading embeddings from /home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt


Reading Embedding File: 2196017it [01:33, 23491.34it/s]


Loaded embeddings from /home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt
Loading embeddings from /home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt


Reading Embedding File: 1703756it [00:52, 32635.36it/s]


Loaded embeddings from /home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt
Loading embeddings from /home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec


...

Coverage checked.
Checking coverage.


Checking Words: 100%|██████████| 37420/37420 [00:00<00:00, 1324949.40it/s]


Coverage checked.
Checking coverage.


Checking Words: 100%|██████████| 37420/37420 [00:00<00:00, 1379957.58it/s]

Coverage checked.
Time taken to clean text: 38080.12386584282 seconds


In [17]:
print(train['full_text'][0])

Many people have car where they live. The thing they don't know is that when you use a car alot of thing can happen like you can get in accidet or the smoke that the car has is bad to breath on if someone is walk but in VAUBAN,Germany they dont have that proble because 70 percent of vauban's families do not own cars,and 57 percent sold a car to move there. Street parkig ,driveways and home garages are forbidden on the outskirts of freiburd that near the French and Swiss borders. You probaly won't see a car in Vauban's streets because they are completely "car free" but If some that lives in VAUBAN that owns a car ownership is allowed,but there are only two places that you can park a large garages at the edge of the development,where a car owner buys a space but it not cheap to buy one they sell the space for you car for $40,000 along with a home. The vauban people completed this in 2006 ,they said that this an example of a growing trend in Europe,The untile states and some where else ar

In [18]:
print(train['clean_text'][0])

many people have car where they live .  the thing they do not know is that when you use a car alot of thing can happen like you can get in accident or the smoke that the car has is bad to breath on if someone is walk but in  , germany they dont have that proble because 70 percent of   '  s families do not own cars , and 57 percent sold a car to move there .  street parking  , driveways and home garages are forbidden on the outskirts of  that near the french and swiss borders .  you probaly will not see a car in   '  s streets because they are completely   "  car free  "   but if some that lives in  that owns a car ownership is allowed , but there are only two places that you can park a large garages at the edge of the development , where a car owner buys a space but it not cheap to buy one they sell the space for you car for  $ 40 , 000 along with a home .  the  people completed this in 2006  , they said that this an example of a growing trend in europe , the untile states and some whe

In [19]:
# Save the cleaned text data to a Parquet file

train.to_parquet('cleaned_train.parquet')

# test.to_parquet('cleaned_test.parquet')

In [ ]:
# save